In [1]:
import numpy as np
import pandas as pd
import gc
from tqdm import tqdm
from os.path import join
import os
import sys
import yaml
from importlib import reload

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [2]:
bos_id = 0
eos_id = 1
pad_id = 2

In [ ]:
MAIL_ID = 1

# Tokenizer [CPU]

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

### Train tokenizer

In [ ]:
tokenizer = Tokenizer(BPE())
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

In [ ]:
special_tokens = ["<bos>", "<eos>", "<pad>"]

trainer = BpeTrainer(
    vocab_size=32_000, 
    special_tokens=special_tokens,
    initial_alphabet=ByteLevel.alphabet()
)

In [ ]:
dataset = pd.read_csv(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN-ES.txt", 
    sep='\t', 
    header = None,
    encoding='utf-8',
)[[0,1]].rename(columns = {0: "EN", 1: "ES"})

In [ ]:
corpus = dataset['EN'].dropna().astype(str).tolist() + dataset['ES'].dropna().astype(str).tolist()
del dataset
gc.collect()

In [ ]:
tokenizer.train_from_iterator(corpus, trainer=trainer)
tokenizer.save("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")

In [ ]:
encoded = tokenizer.encode("Hola! Let's check out this custom BPE.")
encoded.ids, encoded.tokens, tokenizer.decode(encoded.ids)

### Encode data using trained tokenizer

In [ ]:
dataset = pd.read_csv(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN-ES.txt", 
    sep='\t', 
    header = None,
    encoding='utf-8',
)[[0,1]].rename(columns = {0: "EN", 1: "ES"})
dataset = dataset.dropna(how="any")

In [ ]:
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")

In [ ]:
def get_lengths(dataset, batch_size=10_000):
    lengths = []

    for i in tqdm(range(0, len(dataset), batch_size), desc="Measuring lengths"):
        batch = dataset[i: i + batch_size]

        encoded = tokenizer.encode_batch_fast(batch)

        lengths.extend([len(output.ids) for output in encoded])

        del encoded
        gc.collect()

    return lengths

In [ ]:
lengths_en = np.array(get_lengths(dataset["EN"]), dtype=np.uint16)
lengths_es = np.array(get_lengths(dataset["ES"]), dtype=np.uint16)
lengths = np.concat((lengths_en, lengths_es))

In [ ]:
# get length statistics
np.max(lengths), np.percentile(lengths, 99)

In [ ]:
tokenizer = Tokenizer.from_file("/content/drive/MyDrive/machine_translation/datasets/en-es/bpe.json")
max_len = 128

In [ ]:
# get length mask
length_mask = (lengths_en <= max_len) & (lengths_es <= (max_len - 2))   # Decoder input also needs to account for BOS + EOS
length_mask.sum()

In [ ]:
def encode_batch_and_dump(tokenizer, dataset, filename, max_len = 128, batch_size = 10_000, decoder=False):
    tokenizer.enable_padding(
        direction="right", 
        pad_id=pad_id,
        pad_token="<pad>", 
        length=(max_len - 2) if decoder else max_len,   # Decoder input also needs to account for BOS + EOS
    )

    n = dataset.shape[0]

    mmap = np.memmap(
        filename,
        dtype=np.uint16,
        mode="w+",
        shape=(n, max_len),
    )

    for start in tqdm(range(0, n, batch_size), desc="Encoding and dumping"):
        end = min(start + batch_size, n)

        batch = dataset[start:end]
        encoded = tokenizer.encode_batch(batch)

        # print([len(output.ids) for output in encoded if len(output.ids) != max_len])

        matrix_slice = np.asarray(
            [output.ids for output in encoded],
            dtype=np.uint16,
        )

        if decoder:
            mmap[start:end, 1:-1] = matrix_slice
        else:
            mmap[start:end] = matrix_slice

        del batch, encoded, matrix_slice
        gc.collect()

    if decoder:
        mmap[:, 0] = bos_id
        mmap[:, -1] = eos_id

    mmap.flush()
    del mmap
    gc.collect()

In [ ]:
encode_batch_and_dump(
    tokenizer,
    dataset.loc[length_mask, "EN"], 
    "/content/drive/MyDrive/machine_translation/datasets/en-es/EN.npy", 
    max_len=max_len,
    batch_size = 10_000,
)

In [ ]:
encode_batch_and_dump(
    tokenizer,
    dataset.loc[length_mask, "ES"], 
    "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy",
    max_len=max_len,
    batch_size = 10_000,
    decoder=True
)

In [ ]:
a = np.memmap(
    "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy",
    mode="r",
    dtype=np.uint16,
    shape=(length_mask.sum(), max_len)
)
a[:5, -2]

# Train Model [TPU]

In [ ]:
# Make sure all the scripts are in place
# Pull from github

# 1. Define repo details
REPO_URL = "https://github.com/arka816/langlocal.git"
REPO_DIR = "/content/langlocal"

!rm -rf {REPO_DIR}

# 2. Clone if the directory doesn't exist (e.g., after a runtime crash)
if not os.path.exists(REPO_DIR):
    print("Runtime disconnected. Re-cloning scripts...")
    !git clone {REPO_URL}
else:
    # If the runtime didn't crash but you updated code locally, pull the latest changes
    print("Runtime active. Pulling latest script changes...")
    !cd {REPO_DIR} && git pull

# 3. CRITICAL: Add the cloned directory to Python's search path so you can import from it
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

In [ ]:
from machine_translation.transformer.train import train_tpu_single_core, train_gpu

In [ ]:
config_path = join(REPO_DIR, "machine_translation", "transformer", "config.yaml")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

config

In [ ]:
if MAIL_ID == 1:
    src_drive_file = "/content/drive/MyDrive/machine_translation/datasets/en-es/EN.npy"
    tgt_drive_file = "/content/drive/MyDrive/machine_translation/datasets/en-es/ES.npy"
elif MAIL_ID == 2:
    src_drive_file = "/content/drive/MyDrive/EN.npy"
    tgt_drive_file = "/content/drive/MyDrive/ES.npy"


src_file = "/content/EN.npy"
tgt_file = "/content/ES.npy"

if not os.path.exists(src_file):
    !cp {src_drive_file} /content/
if not os.path.exists(tgt_file):
    !cp {tgt_drive_file} /content/

In [ ]:
data_size = config['data']['DATA_SIZE']
sentence_size = config['data']['SENTENCE_SIZE']
file_dims = (data_size, sentence_size)

model_configs = {
    "src_vocab":    config['data']['VOCAB_SIZE'],
    "tgt_vocab":    config['data']['VOCAB_SIZE'],
    "N":            config['model']['NUM_LAYERS'],
    "d_model":      config['model']['EMBEDDING_DIM'], 
    "d_ff":         config['model'].get("FFN_DIM", 4 * config['model']['EMBEDDING_DIM']), 
    "heads":        config['model']['HEADS'],
    "dropout":      config['model']['DROPOUT'], 
    "share_embedding":  config['model']['SHARE_EMBEDDINGS']
}

epochs = config['training']['NUM_EPOCHS']
batch_size = config['training']['BATCH_SIZE']
loader_workers = config['training']['LOADER_WORKERS']
loader_prefetch_factor = config['training']['PREFETCH_FACTOR']

if MAIL_ID == 1:
    checkpoint_filepath = "/content/drive/MyDrive/machine_translation/en-es-machine-translator.pt"
elif MAIL_ID == 2:
    # !cp "/content/drive/MyDrive/en-es-machine-translator.pt" "/content/drive/MyDrive/en-es-machine-translator.pt"
    checkpoint_filepath = "/content/drive/MyDrive/en-es-machine-translator.pt"

dynamic_padding = config['training']['DYNAMIC_PADDING']

In [ ]:
print("Kicking off training...")
train_tpu_single_core(
    src_file,
    tgt_file,
    file_dims,
    model_configs,
    bos_id=bos_id,
    eos_id=eos_id,
    pad_id=pad_id,
    batch_size=batch_size,
    epochs=epochs,
    loader_workers=loader_workers,
    loader_prefetch_factor=loader_prefetch_factor,
    checkpoint_filepath=checkpoint_filepath,
    dynamic_padding=dynamic_padding,
)

In [ ]:
# print("Kicking off training...")
# train_gpu(
#     src_file,
#     tgt_file,
#     file_dims,
#     model_configs,
#     bos_id=bos_id,
#     eos_id=eos_id,
#     pad_id=pad_id,
#     batch_size=batch_size,
#     epochs=epochs,
#     loader_workers=loader_workers,
#     checkpoint_filepath=checkpoint_filepath,
#     dynamic_padding=dynamic_padding
# )

# Run inference [CPU]

In [3]:
sys.path.append("../../")

In [7]:
import inference
reload(inference)
from inference import make_translator

In [8]:
translator = make_translator(bos_id=bos_id, eos_id=eos_id)

Number of parameters:  60524544
Loading checkpoint from C:\Users\arka\Documents\summer 26\llm\machine_translation\cache\en-es-machine-translator.pt...


In [9]:
translator.run("THE COMMISSION OF THE EUROPEAN COMMUNITIES")

torch.return_types.topk(
values=tensor([[-0.7365, -1.1189, -2.4125, -3.1388, -3.6606]],
       grad_fn=<TopkBackward0>),
indices=tensor([[345,   1, 350, 942, 321]]))
torch.return_types.topk(
values=tensor([[-0.5720, -1.2018, -2.2412, -5.0183, -5.1536]],
       grad_fn=<TopkBackward0>),
indices=tensor([[  1, 345, 350, 942, 321]]))
[0, 345, 1]


' que'